# 2. Curated ontology, mapping, temporal KG and reasoning

Runtime POC flow: reviewed Turtle ontology → explicit Northstar source mapping → Semantica `GraphBuilder` → Semantica `TemporalGraphQuery` → Semantica reasoning. The adapter remains the domain-specific ETL boundary because an ontology does not tell us how every JSON record maps into graph entities and edges.

## Load and inspect the curated ontology

In [ ]:
from pathlib import Path
from rdflib import Graph
from northstar_authz.service import load_data

data = load_data()
ontology = Graph()
ontology.parse(Path("ontology/northstar.ttl"), format="turtle")
print("Ontology triples:", len(ontology))

## Map source data and validate mapping coverage

This validation catches undeclared types/properties, invalid endpoints, missing mapped relationship counts, and temporal-field projection mismatches.

In [ ]:
from northstar_authz.adapter import source_to_kg
from northstar_authz.ontology_validation import assert_ontology_mapping_valid

assert_ontology_mapping_valid(data)
graph_input = source_to_kg(data)
print("Entities:", len(graph_input["entities"]))
print("Relationships:", len(graph_input["relationships"]))

## Build the KG with Semantica

In [ ]:
from northstar_authz.service import build_graph

graph = build_graph(data)
print("Graph keys:", graph.keys())
print("Relationship count:", graph.get("num_relationships"))

## Run a point-in-time query with Semantica

Current POC workaround: temporal properties are modelled on reified association individuals in the ontology. The Northstar adapter projects those values onto edge-level `valid_from` / `valid_until` fields for the current temporal query API. This does not demonstrate native temporal filtering over reified association attributes.

In [ ]:
from datetime import datetime, timezone
from northstar_authz.service import temporal_snapshot

before_expiry = temporal_snapshot(graph, datetime(2026, 10, 27, 12, tzinfo=timezone.utc))
at_expiry = temporal_snapshot(graph, datetime(2026, 10, 28, 0, tzinfo=timezone.utc))

def has_ethan_share(snapshot):
    return any(e.get("source") == "Ethan" and e.get("target") == "DOC-107" and e.get("type") == "directShareTo" for e in snapshot.get("relationships", []))

print("Active before expiry:", has_ethan_share(before_expiry))
print("Active at expiry:", has_ethan_share(at_expiry))

## Run authorization reasoning with Semantica

In [ ]:
from northstar_authz.semantica_pipeline import evaluate_with_semantica

as_of = datetime(2026, 11, 1, 12, tzinfo=timezone.utc)
for document_id in ["DOC-101", "DOC-102"]:
    decision = evaluate_with_semantica(data, "Alice", document_id, as_of)
    print(document_id, "->", "ALLOW" if decision.allowed else "DENY")
    print(decision.explanation)


## Migration rule

When the ontology changes, diff old and new Turtle, trace changed terms through the adapter and reasoning/tests, update mapper and validator together, run coverage and regression checks, and require review before replacing the curated schema. See `docs/ontology-validation.md`, ADR-002, and `.claude/skills/ontology-migration/SKILL.md`.

Semantica provides supported KG, temporal-query, and reasoning primitives; the small mapper remains explicit until an ontology-driven source transformation capability is available.